<div dir="rtl">
<h2>بخش اول</h2>
</div>

In [22]:
import cv2 as cv
from pathlib import Path
import plt
import numpy as np
import importlib
import filter
import seg
import char_recognition

noisy_name_postfix = 'noisy'
filtered_name_postfix = 'filtered'
median_denoised_postfix = 'median_denoised'
deblurred_postfix = 'deblurred'
binary_postfix = 'binary'

<h4>1-</h4>
<div dir="rtl">
تصاویر در پوشه Captcha قرار گرفتند
</div>

<h4>2-</h4>
<div dir="rtl">
تصاویر تولیدی (نویزی) در پوشه Captcha قرار گرفتند
</div>

In [23]:
def sp_noise(img, rate = 0.1):
    h, w = img.shape[:2]
    density = h * w * rate
    for i in range(int(density/2)):
        ph = np.random.randint(1, h)
        pw = np.random.randint(1, w)
        img[ph, pw] = 0

    for i in range(int(density/2)):
        ph = np.random.randint(1, h)
        pw = np.random.randint(1, w)
        img[ph, pw] = 255

    return img

for i in range(0, 10):
    img = cv.imread(f'./Captcha/{i}.png')
    gray_img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    noisy_img = sp_noise(gray_img, 0.05)
    plt.imsave(f'./Captcha/{i}_{noisy_name_postfix}.png', noisy_img, cmap='gray')



<h4>3-</h4>
<div dir="rtl">
تصاویر تولیدی (blur شده) در پوشه Captcha قرار گرفتند.
</div>

In [24]:
for i in range(0, 10):
    img = cv.imread(f'./Captcha/{i}_{noisy_name_postfix}.png')
    filtered_img = cv.blur(img, (3,3))
    plt.imsave(f'./Captcha/{i}_{filtered_name_postfix}.png', filtered_img, cmap='gray')

<div dir="rtl">
<h2>بخش دوم</h2>
</div>

<h4>1-</h4>
<div dir="rtl">
از کرنل میانه (median) استفاده می کنیم. خروجی در پوشه Captcha ذخیره شده است.
توضیحات بیشتر در مورد استفاده از این کرنل در گزارش کار آمده است.
</div>

In [25]:
for i in range(0, 10):
    img = cv.imread(f'./Captcha/{i}_{noisy_name_postfix}.png')
    gray_img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    denoised_blurred_img = cv.medianBlur(gray_img, 3)
    plt.imsave(f'./Captcha/{i}_{median_denoised_postfix}.png', denoised_blurred_img, cmap='gray')


<h4>2-</h4>
<div dir="rtl">
توضیحات در گزارش کار
</div>

In [26]:
importlib.reload(filter)

for i in range(0, 10):
    img = cv.imread(f'./Captcha/{i}_{median_denoised_postfix}.png')
    gray_img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    filter.apply_sharpening_filter(gray_img, filter.get_high_boost_filter(7, 2), f'./Captcha/{i}_{deblurred_postfix}.png', 'gray')

<h4>3-</h4>
<div dir="rtl">

</div>

In [27]:
for i in range(0, 10):
    img = cv.imread(f'./Captcha/{i}_{deblurred_postfix}.png')
    bw_img = cv.threshold(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 127.5, 255, cv.THRESH_BINARY_INV)[1]
    plt.imsave(f'./Captcha/{i}_{binary_postfix}.png', bw_img, cmap='gray')

<div dir="rtl">
<h2>بخش سوم</h2>
</div>

<h4>1, 2-</h4>
<div dir="rtl">
توضیحات در گزارش کار
</div>

In [29]:
importlib.reload(seg)

for i in range(0, 10):
    img = cv.imread(f'./Captcha/{i}_{binary_postfix}.png')
    g = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    bw = cv.threshold(g, 127.5, 255, cv.THRESH_BINARY)[1]

    value_count = seg.segment(bw, i)
    print(f'Value count for file {i} is: {value_count}')

Value count for file 0 is: 3
Value count for file 1 is: 3
Value count for file 2 is: 3
Value count for file 3 is: 2
Value count for file 4 is: 3
Value count for file 5 is: 3
Value count for file 6 is: 2
Value count for file 7 is: 3
Value count for file 8 is: 3
Value count for file 9 is: 3


<div dir="rtl">
<h2>بخش چهارم</h2>
</div>

<div dir="rtl">
فایل خروجی ComparisonResult ساخته شد
</div>

In [30]:
importlib.reload(char_recognition)

mapset_pth = './Mapset'
seg_output_pth = './Captcha/Segmentation'

mapset_dir = Path(mapset_pth)
seg_output_dir = Path(seg_output_pth)

map_result_name = ''
for i, seg_output_file in enumerate(seg_output_dir.glob('*.png')):
    img1 = cv.imread(seg_output_file.as_posix())
    img1 = cv.cvtColor(img1, cv.COLOR_BGR2GRAY)
    sim_rate = 0
    for map_file in mapset_dir.glob('*.png'):
        img2 = cv.imread(map_file.as_posix())
        img2 = cv.cvtColor(img2, cv.COLOR_BGR2GRAY)
        sr = char_recognition.similarity(img1, img2)
        if sr > sim_rate:
            sim_rate = sr
            map_result_name = map_file.name

    with open(f'ComparisonResult.csv', 'a', encoding= 'utf-8') as f:
        if i == 0:
            f.write('output file, map file, similarity rate\n')
        f.write(f'{seg_output_file.name}, {map_result_name}, %{round(sim_rate * 100, 1)}\n')